In [1]:
import pandas as pd

# 1. Load the core datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Load the supporting metadata / drivers
stores_df = pd.read_csv('stores.csv')
oil_df = pd.read_csv('oil.csv')
holidays_df = pd.read_csv('holidays_events.csv')
transactions_df = pd.read_csv('transactions.csv')

# Quick check on dimensions to verify they loaded correctly
print(f"train.csv shape: {train_df.shape}")
print(f"test.csv shape: {test_df.shape}")
print(f"stores.csv shape: {stores_df.shape}")
print(f"oil.csv shape: {oil_df.shape}")
print(f"holidays_events.csv shape: {holidays_df.shape}")
print(f"transactions.csv shape: {transactions_df.shape}")

train.csv shape: (3000888, 6)
test.csv shape: (28512, 5)
stores.csv shape: (54, 5)
oil.csv shape: (1218, 2)
holidays_events.csv shape: (350, 6)
transactions.csv shape: (83488, 3)


In [2]:
# 1. Check columns and data types for our core and supporting tables
print("--- TRAIN INFO ---")
print(train_df.info())

print("\n--- STORES INFO ---")
print(stores_df.head(2))

print("\n--- OIL INFO ---")
print(oil_df.info())

print("\n--- TRANSACTIONS INFO ---")
print(transactions_df.info())

--- TRAIN INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB
None

--- STORES INFO ---
   store_nbr   city      state type  cluster
0          1  Quito  Pichincha    D       13
1          2  Quito  Pichincha    D       13

--- OIL INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB
None

--- TRANSACTIONS INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries

In [3]:
# 1. Convert 'date' columns to actual datetime objects across all tables
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])
oil_df['date'] = pd.to_datetime(oil_df['date'])
transactions_df['date'] = pd.to_datetime(transactions_df['date'])
holidays_df['date'] = pd.to_datetime(holidays_df['date'])

# 2. Forward-fill missing oil prices (weekends/holidays)
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].ffill()

# (Optional edge case check: if the very first row of oil has a NaN, backfill it)
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].bfill()

print("Dates converted and oil prices cleaned successfully!")

Dates converted and oil prices cleaned successfully!


In [4]:
# Verify data types and check for any remaining nulls in oil
print("--- VERIFIED DATETIME TYPES ---")
print("train date dtype:", train_df['date'].dtype)
print("oil date dtype:", oil_df['date'].dtype)
print("transactions date dtype:", transactions_df['date'].dtype)

print("\n--- CHECKING OIL NULLS AFTER FILL ---")
print("Remaining null oil values:", oil_df['dcoilwtico'].isnull().sum())

--- VERIFIED DATETIME TYPES ---
train date dtype: datetime64[ns]
oil date dtype: datetime64[ns]
transactions date dtype: datetime64[ns]

--- CHECKING OIL NULLS AFTER FILL ---
Remaining null oil values: 0


In [5]:
# 1. Merge store metadata on store_nbr
train_merged = pd.merge(train_df, stores_df, on='store_nbr', how='left')

# 2. Merge daily oil prices on date
train_merged = pd.merge(train_merged, oil_df, on='date', how='left')

# 3. Merge store transactions on date and store_nbr
train_merged = pd.merge(train_merged, transactions_df, on=['date', 'store_nbr'], how='left')

# Verify the final shape and view the first few rows
print(f"Original train shape: {train_df.shape}")
print(f"Master merged train shape: {train_merged.shape}")
display(train_merged.head(3))

Original train shape: (3000888, 6)
Master merged train shape: (3000888, 12)


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico,transactions
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,93.14,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,93.14,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,93.14,NaN


In [6]:
# Check all available columns in our master training table
print("Columns in train_merged:")
print(train_merged.columns.tolist())

# Display a couple of sample rows to inspect the merged values
display(train_merged.head(2))

Columns in train_merged:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'transactions']


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico,transactions
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,93.14,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,93.14,NaN


In [7]:
# Print each column name on a clean new line
for col in train_merged.columns:
    print(col)

id
date
store_nbr
family
sales
onpromotion
city
state
type
cluster
dcoilwtico
transactions


In [8]:
# 1. Load the holidays dataset (adjust the filename if your path is different)
holidays_events = pd.read_csv('holidays_events.csv')

# 2. Inspect its shape and first few rows
print(f"Holidays shape: {holidays_events.shape}")
display(holidays_events.head(5))

Holidays shape: (350, 6)


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [9]:
# Check unique types of events and geographic scopes
print("Event Types:")
print(holidays_events['type'].value_counts())
print("\nGeographic Scopes (Locale):")
print(holidays_events['locale'].value_counts())
print("\nTransferred Status:")
print(holidays_events['transferred'].value_counts())

Event Types:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

Geographic Scopes (Locale):
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

Transferred Status:
transferred
False    338
True      12
Name: count, dtype: int64


In [10]:
# 1. Drop transferred holidays (they didn't happen on this date)
holidays_clean = holidays_events[holidays_events['transferred'] == False].copy()

# 2. Split into National, Regional, and Local scopes
national_holidays = holidays_clean[holidays_clean['locale'] == 'National'][['date', 'type', 'description']]
national_holidays['is_holiday'] = 1

regional_holidays = holidays_clean[holidays_clean['locale'] == 'Regional'][['date', 'locale_name', 'type', 'description']]
regional_holidays.rename(columns={'locale_name': 'state'}, inplace=True)
regional_holidays['is_regional_holiday'] = 1

local_holidays = holidays_clean[holidays_clean['locale'] == 'Local'][['date', 'locale_name', 'type', 'description']]
local_holidays.rename(columns={'locale_name': 'city'}, inplace=True)
local_holidays['is_local_holiday'] = 1

print(f"Cleaned National holidays: {len(national_holidays)}")
print(f"Cleaned Regional holidays: {len(regional_holidays)}")
print(f"Cleaned Local holidays: {len(local_holidays)}")

Cleaned National holidays: 166
Cleaned Regional holidays: 24
Cleaned Local holidays: 148


In [11]:
## 1. Ensure the date column in all holiday subsets is converted to datetime
national_holidays['date'] = pd.to_datetime(national_holidays['date'])
regional_holidays['date'] = pd.to_datetime(regional_holidays['date'])
local_holidays['date'] = pd.to_datetime(local_holidays['date'])

# 2. Merge National holidays (matches on date only)
train_merged = pd.merge(train_merged, national_holidays[['date', 'is_holiday']], on='date', how='left')

# 3. Merge Regional holidays (matches on date and state)
train_merged = pd.merge(train_merged, regional_holidays[['date', 'state', 'is_regional_holiday']], on=['date', 'state'], how='left')

# 4. Merge Local holidays (matches on date and city)
train_merged = pd.merge(train_merged, local_holidays[['date', 'city', 'is_local_holiday']], on=['date', 'city'], how='left')

# 5. Fill missing values with 0 (0 = no holiday for that scope on that day)
train_merged['is_holiday'] = train_merged['is_holiday'].fillna(0)
train_merged['is_regional_holiday'] = train_merged['is_regional_holiday'].fillna(0)
train_merged['is_local_holiday'] = train_merged['is_local_holiday'].fillna(0)

# 6. Create a combined binary indicator for "any holiday affecting this store"
train_merged['any_holiday'] = ((train_merged['is_holiday'] == 1) |
                               (train_merged['is_regional_holiday'] == 1) |
                               (train_merged['is_local_holiday'] == 1)).astype(int)

print(f"train_merged shape after holiday merges: {train_merged.shape}")
print(train_merged['any_holiday'].value_counts())

train_merged shape after holiday merges: (3008280, 16)
any_holiday
0    2746128
1     262152
Name: count, dtype: int64


In [12]:
# 1. Merge store metadata on store_nbr
test_merged = pd.merge(test_df, stores_df, on='store_nbr', how='left')

# 2. Merge daily oil prices on date
test_merged = pd.merge(test_merged, oil_df, on='date', how='left')

# 3. Merge National holidays (matches on date only)
test_merged = pd.merge(test_merged, national_holidays[['date', 'is_holiday']], on='date', how='left')

# 4. Merge Regional holidays (matches on date and state)
test_merged = pd.merge(test_merged, regional_holidays[['date', 'state', 'is_regional_holiday']], on=['date', 'state'], how='left')

# 5. Merge Local holidays (matches on date and city)
test_merged = pd.merge(test_merged, local_holidays[['date', 'city', 'is_local_holiday']], on=['date', 'city'], how='left')

# 6. Fill missing holiday values with 0
test_merged['is_holiday'] = test_merged['is_holiday'].fillna(0)
test_merged['is_regional_holiday'] = test_merged['is_regional_holiday'].fillna(0)
test_merged['is_local_holiday'] = test_merged['is_local_holiday'].fillna(0)

# 7. Create combined binary indicator for any holiday affecting this store
test_merged['any_holiday'] = ((test_merged['is_holiday'] == 1) |
                               (test_merged['is_regional_holiday'] == 1) |
                               (test_merged['is_local_holiday'] == 1)).astype(int)

print(f"Original test shape: {test_df.shape}")
print(f"Master merged test shape: {test_merged.shape}")
print(test_merged.columns.tolist())

Original test shape: (28512, 5)
Master merged test shape: (28512, 14)
['id', 'date', 'store_nbr', 'family', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'is_holiday', 'is_regional_holiday', 'is_local_holiday', 'any_holiday']


In [13]:
# Compare column names and structures side by side
train_cols = set(train_merged.columns)
test_cols = set(test_merged.columns)

print(f"Total columns in train_merged: {len(train_cols)}")
print(f"Total columns in test_merged: {len(test_cols)}")

# Find any columns unique to one set or the other
print("\nColumns in train_merged but NOT in test_merged:")
print(train_cols - test_cols)

print("\nColumns in test_merged but NOT in train_merged:")
print(test_cols - train_cols)

Total columns in train_merged: 16
Total columns in test_merged: 14

Columns in train_merged but NOT in test_merged:
{'sales', 'transactions'}

Columns in test_merged but NOT in train_merged:
set()


In [14]:
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

# 1. Define features to drop (IDs, dates, target, and transactions since it's missing in test)
drop_cols = ['id', 'date', 'sales', 'transactions']
features = [col for col in train_merged.columns if col not in drop_cols]
target = 'sales'

# 2. Convert categorical columns to 'category' dtype so XGBoost can handle them natively
cat_cols = ['family', 'city', 'state', 'type']
for col in cat_cols:
    train_merged[col] = train_merged[col].astype('category')
    test_merged[col] = test_merged[col].astype('category')

# 3. Create a time-based validation split
# Let's use the last 60 days of the training data as our local validation set
max_date = train_merged['date'].max()
val_cutoff = max_date - pd.Timedelta(days=60)

train_mask = train_merged['date'] < val_cutoff
val_mask = train_merged['date'] >= val_cutoff

X_train = train_merged.loc[train_mask, features]
y_train = train_merged.loc[train_mask, target]

X_val = train_merged.loc[val_mask, features]
y_val = train_merged.loc[val_mask, target]

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")

Training set shape: (2899578, 12)
Validation set shape: (108702, 12)


In [15]:
# Initialize the XGBoost Regressor with categorical support enabled
model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

# Train the model
print("Training baseline model...")
model.fit(X_train, y_train)

# Predict and evaluate on the validation set
val_preds = model.predict(X_val)
# Sales can't be negative, so clip predictions at 0
val_preds = val_preds.clip(min=0)

rmse = root_mean_squared_error(y_val, val_preds)
print(f"\nValidation RMSE: {rmse:.4f}")

Training baseline model...

Validation RMSE: 308.9290


In [16]:
# 1. Save the test 'id' column safely for our final submission
test_ids = test_merged['id']

# 2. Extract the exact same feature columns from test_merged that we used for training
X_test = test_merged[features]

# 3. Predict sales using our trained baseline model
print("Generating predictions on the test set...")
test_preds = model.predict(X_test)

# 4. Sales cannot be negative, so clip anything below 0
test_preds = test_preds.clip(min=0)

# 5. Create the submission dataframe matching Kaggle's required format (id and sales)
submission = pd.DataFrame({
    'id': test_ids,
    'sales': test_preds
})

# 6. Save to a CSV file
submission.to_csv('submission_baseline.csv', index=False)

print("Submission file 'submission_baseline.csv' created successfully!")
display(submission.head())

Generating predictions on the test set...
Submission file 'submission_baseline.csv' created successfully!


,id,sales
0,3000888,3.019371
1,3000889,0.000000
2,3000890,3.409735
3,3000891,2306.182617
4,3000892,0.000000
